# Pandas et les données textuelles
.str et chaines de caractères

User guide : https://pandas.pydata.org/docs/user_guide/10min.html  

Cheat Sheet : https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf

Complete course : https://stefaniemolin.com/pandas-workshop

## Rappels sur les chaînes de caractères

## Rappels sur Pandas

In [ ]:
# charger les bibliothèques requises
import pandas as pd

# Charger des données locales
df = pd.read_csv("../data/css_openalex_26022026.csv")

# Ou charger des données avec pandas depuis une URL
data_url = "https://raw.githubusercontent.com/pyshs/URFIST-Lyon-2026/refs/heads/main/data/css_openalex_26022026.csv"
df_url = pd.read_csv(data_url)

## Texte : les fonctions utiles(-ish)

- **Compter des caractères, joindre des textes**
  - Series.str.len() : Longueur des chaînes de caractères.
  - Series.str.join() : Concaténer des chaînes.
- **Manipuler la casse**
  - Series.str.lower() : Convertir le texte en minuscules.
  - Series.str.upper() : Convertir le texte en majuscules.
- **Manipuler les espaces et séparer des chaines**
  - Series.str.strip() : Supprimer les espaces en début et fin de chaîne.
  - Series.str.split() : Diviser une chaîne de caractères en une liste
- **Remplacer des éléments**
  - Series.str.replace() : Remplacer des caractères ou des mots
- **Rechercher / compter / extraire**
  - Series.str.contains() : Vérifier si une sous-chaîne est présente.
  - Series.str.count() : Compter des occurences 
  - Series.str.extract() : Extraire des motifs avec des expressions régulières.
- **Passe décisive pour émilien et la suite**
  - Series.str.get_dummies()
- **Appliquer des fonctions perso**
  - DataFrame.apply() : Appliquer une fonction personnalisée à une colonne (ex : calculer la longueur moyenne des textes).


## Le champ des possibles

In [ ]:
# Jouer avec la casse

# exemples tout prêts à l'emploi :
# Courant :
# df["title"].str.upper()
# df["title"].str.lower()

# Plus rare :
# df["title"].str.capitalize()
# df["title"].str.title()
# df["title"].str.swapcase()


In [ ]:
# Jouer avec les espaces

# df["title"].str.strip()  # Supprimer les espaces en début et fin de chaîne

## Plus rare :
# df["title"].str.lstrip()  # Supprimer les espaces à gauche
# df["title"].str.rstrip()  # Supprimer les espaces à droite

In [ ]:
# Jouer avec les remplacements

# df["title"].str.replace(" ", "_")  # Remplacer les espaces par des underscores
# df["title"].str.replace(" ", "")  # Supprimer les espaces


In [ ]:
# Jouer avec les séparations

# When set to None (the default value), will split on any whitespace
#    character (including \\n \\r \\t \\f and spaces) and will discard
#    empty strings from the result.

# df["title"].str.split()  # Séparer
# df["title"].str.split(" ")  # Séparer en utilisant l'espace


In [ ]:
# Jouer avec les recherches
# (plus de précisions après avec les Regex)

## Courant :
# df["title"].str.contains("Python")  # Vérifier si "Python" est dans le texte
# df["title"].str.count("Python")  # Compter le nombre d'occurrences

## Plus rare :
# df["title"].str.startswith("C")  # Vérifier si le texte commence par "C"
# df["title"].str.endswith("e")  # Vérifier si le texte se termine par "e"

## Plus niche : On voit ça plus bas avec les regex
# df["title"].str.extract()
# df["title"].str.find()
# df["title"].str.findall()



## But Why ?

### Hacktime 1
- enlever les crochets et parenthèses des champs textes
- regrouper titre et abstract
- calculer la longueur du texte et sa distribution

In [ ]:
# Notre code ici


In [ ]:
# Avec une fonction et apply

# def …(…):
#     """
#     …ma doc blabla…
#     """
#     … = str(…)
#     … = ["[", "]", "(", ")"]
#     for … in …:
#         … = ….replace(…, "")
#     return …

# df["…"].apply(str).apply(…)

# nb : vous pouvez supprimer les NaN si plus simple :
# df = df.dropna(subset=["title", "abstract"])


### Hacktime 2
- identifier les textes qui mentionnent "machine learning"
- calculer combien de fois chaque texte mentionne "computational" 
- comparer la distribution de computational entre les articles ML et les autres
  - ex : grouper selon qu'ils sont ML ou non et utiliser describe

In [ ]:
# Notre code ici


### Solutions

In [ ]:
# PROPOSITION SOLUTION HACKTIME 1

df["title"] = (
    df["title"]
    .str.replace("[", "")
    .str.replace("]", "")
    .str.replace("(", "")
    .str.replace(")", "")
)
df["abstract"] = (
    df["abstract"]
    .str.replace("[", "")
    .str.replace("]", "")
    .str.replace("(", "")
    .str.replace(")", "")
)
# Regouper le titre et l'abstract
df["texte"] = df["title"] + "\n" + df["abstract"]

# Calucler la taille du texte
df["taille"] = df["texte"].str.len()

# Visualiser la distribution des tailles
df["taille"].hist(bins=100)  # j'aime pas leur nouvelle syntaxe ?

# NB : il faudrait filtrer les textes trop longs, etc.

In [ ]:
# PROPOSITION SOLUTION HACKTIME 2
df["ML"] = df["texte"].str.lower().str.contains("machine learning")
df["nb_computational"] = df["texte"].str.lower().str.count("computational")
df.groupby("ML")["nb_computational"].describe().T  # T pour transposer

# "ça ne me sert à rien" vibe pour ce résultat

In [ ]:
# Alternatives nettoyage:

# # NB On peut se faciliter la vie avec une regex :
# df["title"] = df["title"].str.replace(r"[\[\]']", "", regex=True)
# df["abstract"] = df["abstract"].str.replace(r"[\[\]']", "", regex=True)

# ====================

# # Ou créer une fonction et l'appliquer
# def nettoyer(texte):
#     """
#     Fonction de nettoyage pour supprimer crochets et parenthèses
#     """
#     if isinstance(texte, str):
#         return texte.replace("[", "").replace("]", "").replace("(", "").replace(")", "")
#     return texte


# df["title"] = df["title"].apply(nettoyer)
# df["abstract"] = df["abstract"].apply(nettoyer)

# Possible soucis des NaN, d'où la vérification du type dans la fonction de nettoyage.


## Vive les Regex

- Et si je voulais "artificial intelligence" OU "machine learning" ?
- Et s'il fallait extraire le mot computational dans son contexte (X caractères avant/après)
- Et si je voulais extraire toutes les dates dans le texte ?
- Et si et si et si…

Les regex sont là pour ~~vous aider~~ vous faire souffrir !!

L'idée : un motif (pattern) un motif qui permet de rechercher, extraire ou remplacer des chaînes de caractères selon des règles précises.

Par ici : https://regex101.com/
ou du côté doc python : https://docs.python.org/3/library/re.html

NB : il y a différentes "flavors" de regex

### Syntaxe de base :

| Symbole | Description |
|---------|-------------|
| .       | n’importe quel caractère |
| *       | 0 ou plusieurs fois |
| +       | 1 ou plusieurs fois |
| ?       | 0 ou 1 fois |
| [abc]   | un caractère parmi a, b ou c |
| [^abc]  | un caractère sauf a, b ou c |
| \d      | un chiffre ([0‑9]) |
| \w      | un caractère alphanumérique |
| \s      | un espace blanc |
| \b      | limite de mots |
| ^       | début de ligne |
| $       | fin de ligne |
| ( )     | groupe capturant |


Exemples : 
| Cas              | Motif |
|------------------|-------|
| Trouver un email (meh) | \b[\w.-]+@[\w.-]+\.\w+\b |
| Extraire une année | \b\d{4}\b |
| "machine learning" ou "artificial intelligence" | machine learning\|artificial intelligence |
| Python avec ou sans majuscule (début de mot) | \b[Pp]ython |

In [ ]:
# utile, mais ça casse les tables markdown ???

# {n}    | exactement n fois |
# {n,m}  | entre n et m fois |
# {n,}   | au moins n fois |

### Les patterns

**Exemple :** Extraire le mot "computational" et son contexte (20 caractères avant/après)

In [ ]:
import re

re.findall(
    r"(.{20}\bcomputational.{20})",
    "J'ai des gros data et j'aime le computational power blabla blabla bla",
)

# .{0,20} -> si on veut 0 à 20 caractères

### En pandas : 
str.find() # identifie la position
str.extract() # extrait le premier match
str.findall() # tous les matchs

In [ ]:
# df["title"].str.extract()
# df["title"].str.find()
# df["title"].str.findall()

In [ ]:
# la position de "computational" dans le texte
df["texte"].str.lower().str.find(r"computational)")

In [ ]:
# les 20 caractères avant et après "computational"
df["texte"].str.lower().str.findall(r"(.{0,20}computational.{0,20})")

In [ ]:
# 1ère occurence des 20 caractères avant et après "computational"
# renvoie un df par défaut, expand=False pour une série
df["texte"].str.lower().str.extract(r"(.{0,20}computational.{0,20})")

### Bac a sable regex

In [ ]:
re.findall(r"([0-9]{4})", "Ceci est une année : 1990 ; Ceci en est une autre 2000")

masque = r"([0-9]{2}\s+(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4})"
re.findall(masque, "It is the 21 September 2020 , we are all happy")

In [ ]:
c = re.compile(r"\bmodel\b", re.I)  # re.I pour ignorer la casse
c.findall("This is a Model.")

In [ ]:
df["model"] = df["texte"].apply(lambda x: bool(c.search(str(x))))
df["model"].value_counts()

### Hacktime 3
- Extraire le premier mot de chaque texte
- Extraire la première année de chaque texte (si elle existe)
- Compter combien de textes contiennent au moins une année

In [ ]:
# Notre code ici


In [ ]:
# Solution hacktime 3

# 1. extraire le premier mot du titre
df["premier_mot"] = df["texte"].str.extract(r"^(\w+)")

# 2. extraire la première année de chaque texte (si elle existe)
df["annee"] = df["texte"].str.extract(r"(\b[0-9]{4}\b)")

# affichage pour vérifier
print(df[["title", "premier_mot", "annee"]].head())

# 3. compter combien de texte contiennent une année
print(f"{df['annee'].notna().sum()} textes contiennent au moins une année")